In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep
import base64

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    print(f"🔁 Using token #{token_index + 1}")
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === Paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step3_android_detection_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step4_keyword_check_output.csv"

df = pd.read_csv(input_path)

updated_name = []
updated_description = []
updated_topics = []
updated_readme_flags = []
keyword_checks = []

def search_android_in_readme_like(repo):
    # Try README first
    readme_url = f"https://api.github.com/repos/{repo}/readme"
    r = requests.get(readme_url, headers=get_headers())
    if r.status_code == 200:
        content = r.json().get("content", "")
        try:
            decoded = base64.b64decode(content).decode("utf-8", errors="ignore").lower()
            if "android" in decoded:
                return True
        except Exception:
            pass

    # Fallback to other repo contents (searching .md or .rst)
    contents_url = f"https://api.github.com/repos/{repo}/contents"
    r = requests.get(contents_url, headers=get_headers())
    if r.status_code == 200:
        for item in r.json():
            name = item.get("name", "").lower()
            if name.endswith((".md", ".markdown", ".rst")):
                file_url = item.get("download_url")
                if file_url:
                    try:
                        content = requests.get(file_url, headers={"User-Agent": "android-repo-crawler/1.0"}).text.lower()
                        if "android" in content:
                            return True
                    except Exception:
                        continue

    # Try GitHub Wiki (Home.md)
    wiki_url = f"https://raw.githubusercontent.com/wiki/{repo}/Home.md"
    r = requests.get(wiki_url, headers={"User-Agent": "android-repo-crawler/1.0"})
    if r.status_code == 200 and "android" in r.text.lower():
        return True

    return False

# === Main Loop ===
for i, row in df.iterrows():
    if row["status"] != "pass":
        updated_name.append(row["name"])
        updated_description.append(row["description"])
        updated_topics.append(row["topics"])
        updated_readme_flags.append(row["android_in_readme"])
        keyword_checks.append("reject")
        continue

    repo = row["full_name"]
    name = row.get("name", "")
    desc = row.get("description", "")
    topics_raw = row.get("topics", "")
    topics_list = [t.strip().lower() for t in str(topics_raw).split(",") if t.strip()]

    # Step 1: Real-time metadata
    metadata_url = f"https://api.github.com/repos/{repo}"
    r = requests.get(metadata_url, headers=get_headers())
    if r.status_code == 200:
        metadata = r.json()
        name = str(metadata.get("name", "")).lower()
        desc = str(metadata.get("description", "")).lower()
    updated_name.append(name)
    updated_description.append(desc)

    # Step 2: Real-time topics
    topics_url = f"https://api.github.com/repos/{repo}/topics"
    r = requests.get(topics_url, headers=get_headers())
    if r.status_code == 200:
        topic_names = r.json().get("names", [])
        topics_list = [t.lower() for t in topic_names]
        updated_topics.append(",".join(topic_names))
    else:
        updated_topics.append(topics_raw)

    # Step 3: Real-time README or similar
    in_readme = search_android_in_readme_like(repo)
    updated_readme_flags.append("yes" if in_readme else "no")

    # Step 4: Keyword validation
    found = (
        "android" in name or
        "android" in desc or
        any("android" in t for t in topics_list) or
        in_readme
    )
    keyword_checks.append("pass" if found else "reject")

    if i % 100 == 0:
        print(f"🔎 Checked {i+1} repos...")

# === Save updated output ===
df["name"] = updated_name
df["description"] = updated_description
df["topics"] = updated_topics
df["android_in_readme"] = updated_readme_flags
df["keyword_check"] = keyword_checks

df.to_csv(output_path, index=False)
print(f"✅ Step 4 complete. Saved to: {output_path}")
